## RAG Application Using Typesense

In [5]:
import typesense

In [13]:
from dotenv import load_dotenv
import os

load_dotenv()

client = typesense.Client({
    "nodes": [
        {
            "host": "zryleipkv0htd7a2p-1.a1.typesense.net",
            "port": "443",
            "protocol": "https",
        }
    ],
    "api_key": os.getenv("TYPESENSE_API_KEY"),
    "connection_timeout_seconds": 2,
})

books_schema = {
    "name": "books",
    "fields": [
        {"name": "title", "type": "string"},
        {"name": "authors", "type": "string[]", "facet": True},
        {"name": "publication_year", "type": "int32", "facet": True},
        {"name": "ratings_count", "type": "int32"},
        {"name": "average_rating", "type": "float"},
    ],
    "default_sorting_field": "ratings_count",
}

print(client.collections.create(books_schema))

{'created_at': 1793107822, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': T

In [12]:
client

In [14]:
with open("books.jsonl", "r", encoding="utf-8") as json_file:
    books = json_file.read()
    client.collections["books"].documents.import_(books)

In [16]:
search_parameters={
    'q': "harry potter",
    'query_by': "title, authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [17]:
search_parameters = {
    "q": "harry potter",
    "query_by": "title,authors",
    "filter_by": "publication_year:<1998",
    "sort_by": "publication_year:desc",
}

client.collections["books"].documents.search(search_parameters)


{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name

In [ ]:
search_parameters = {
    "q": "experiment",
    "query_by": "title,authors",
    "facet_by": "authors",
    "sort_by": "average_rating:desc",
}

client.collections["books"].documents.search(search_parameters)


In [33]:
### Langchain + Typsense + Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### PDF Loader

In [ ]:

## load all the pdf files from the directory
dir_loader=DirectoryLoader(
    "data/pdf",
    glob="**/*.pdf", ##Pattern to match files
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False
)
pdf_documents = dir_loader.load()

print(f"Loaded {len(pdf_documents)} pages")
print(pdf_documents[0].page_content)
print(pdf_documents[0].metadata)

### Split documents into chunks

In [34]:
def split_documents(documents):
    text_splitter = text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=100,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
        )

    split_docs = text_splitter.split_documents(documents)

    print(
        f"Split {len(documents)} documents into {len(split_docs)} chunks"
    )

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks = split_documents(
    pdf_documents,
)
embeddings = HuggingFaceEmbeddings()

In [ ]:
chunks

In [39]:
docsearch = Typesense.from_documents(
    chunks,
    embeddings,
    typesense_client_params={
        "host": "zryleipkv0htd7a2p-1.a1.typesense.net",
        "port": "443",
        "protocol": "https",
        "typesense_api_key": os.getenv("TYPESENSE_API_KEY"),
        "typesense_collection_name": "lang-chain",
    },
)

In [ ]:
query = "What is my name?"
found_docs = docsearch.similarity_search(query)
print(found_docs)

In [ ]:
### Retriever
retriever = docsearch.as_retriever()
retriever

In [50]:
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY is missing from your .env file")
llm = ChatGroq(api_key=groq_api_key, model="openai/gpt-oss-20b", temperature=0.1, max_tokens=500)

query = "What is my first project that i have build and what it does?"
context=retriever.invoke(query)[0]
prompt=f"""Use the following context to answer the question concisely and accurately 
elaborate little bit don't give direct answers. Context: {context.page_content} Question: {query} Answer: """ 

response = llm.invoke(prompt.format(prompt))

print(response.content)

Your first project is **Ai‑side‑builder**. It’s an intelligent web‑creation platform that lets you turn a concept into a live website in just minutes. You simply describe what you want, see an instant preview, and then fine‑tune the design with straightforward requests—so you can focus on the idea rather than the technical details.
